In [12]:
import numpy as np 
import pandas as pd 
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "mbart" in root.lower() and f == "model.safetensors":
            print(os.path.join(root, f))

/kaggle/input/datasets/jlkeyx/cultural-recipes-checkpoint/cultural-recipes-mbart/checkpoint-19830/model.safetensors
/kaggle/input/datasets/jlkeyx/cultural-recipes-checkpoint/cultural-recipes-mbart/final/model.safetensors


In [10]:
!pip install -q transformers datasets sentencepiece accelerate
!pip install -q rouge-score sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00


In [ ]:
#Loading Tokenizer and Model
from transformers import MBart50TokenizerFast
import torch

MODEL_NAME = "facebook/mbart-large-50"
tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Tokenizer type: {type(tokenizer).__name__}")  # should say MBart50TokenizerFast
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)  
# model = model.to("cuda")                                    


In [ ]:
#Loading full dataset

from datasets import Dataset, DatasetDict
import json, os

DATA_ROOT = "/kaggle/input/datasets/jlkeyx/cultural-recipes-nlp"

def load_flat_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

raw_datasets = {}
for direction in ["cn2en", "en2cn"]:
    train = load_flat_jsonl(f"{DATA_ROOT}/recipe_match_{direction}_train.jsonl")
    valid = load_flat_jsonl(f"{DATA_ROOT}/recipe_match_{direction}_valid.jsonl")
    test  = load_flat_jsonl(f"{DATA_ROOT}/recipe_match_{direction}_test.jsonl")

    raw_datasets[f"recipe_match_{direction}"] = DatasetDict({
        "train": Dataset.from_list(train),
        "valid": Dataset.from_list(valid),
        "test":  Dataset.from_list(test),
    })
    print(f"recipe_match_{direction}: train={len(train):,}, "
          f"valid={len(valid):,}, test={len(test):,}")


In [ ]:
#mBART Tokenization

MAX_INPUT_LENGTH  = 512
MAX_TARGET_LENGTH = 512

def make_tokenize_fn(src_lang, tgt_lang):
    def tokenize(batch):
        # Set source language
        tokenizer.src_lang = src_lang
        
        model_inputs = tokenizer(
            batch["source"],
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
        )
        
        # setting the target language
        tokenizer.src_lang = tgt_lang
        labels = tokenizer(
            batch["target"],
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
        )
        
        tokenizer.src_lang = src_lang
        
        labels["input_ids"] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label]
            for label in labels["input_ids"]
        ]
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return tokenize

tokenized = {}

# cn2en --> source=Chinese, target=English
print("Tokenizing recipe_match_cn2en...")
tokenized["recipe_match_cn2en"] = raw_datasets["recipe_match_cn2en"].map(
    make_tokenize_fn(src_lang="zh_CN", tgt_lang="en_XX"),
    batched=True,
    batch_size=64,
    remove_columns=["source", "target"],
)
print(f"Done: {tokenized['recipe_match_cn2en']}")

# en2cn --> source=English, target=Chinese
print("Tokenizing recipe_match_en2cn...")
tokenized["recipe_match_en2cn"] = raw_datasets["recipe_match_en2cn"].map(
    make_tokenize_fn(src_lang="en_XX", tgt_lang="zh_CN"),
    batched=True,
    batch_size=64,
    remove_columns=["source", "target"],
)
print(f"Done: {tokenized['recipe_match_en2cn']}")

In [ ]:
#This cell combines en2cn and cn2en

from datasets import concatenate_datasets

combined = DatasetDict({
    "train": concatenate_datasets([
        tokenized["recipe_match_cn2en"]["train"],
        tokenized["recipe_match_en2cn"]["train"],
    ]),
    "valid": concatenate_datasets([
        tokenized["recipe_match_cn2en"]["valid"],
        tokenized["recipe_match_en2cn"]["valid"],
    ]),
    "test": concatenate_datasets([
        tokenized["recipe_match_cn2en"]["test"],
        tokenized["recipe_match_en2cn"]["test"],
    ]),
})

print(f"Combined dataset:")
print(f"  train: {len(combined['train']):,} pairs")
print(f"  valid: {len(combined['valid']):,} pairs")
print(f"  test:  {len(combined['test']):,} pairs")

In [ ]:
#Note: this is an example for  most recent mBART directory. this cell needs
#to be edited for each checkpoint update

import torch, gc, os, shutil
from transformers import (MBartForConditionalGeneration,
                          Seq2SeqTrainer,
                          DataCollatorForSeq2Seq)
from torch.utils.data import DataLoader

MBART_CHECKPOINT_BASE = "/kaggle/input/datasets/jlkeyx/cultural-recipes-checkpoint/cultural-recipes-mbart"
#kaggle/input/datasets/jlkeyx/cultural-recipes-checkpoint/cultural-recipes-mbart
RESUME_CHECKPOINT = os.path.join(MBART_CHECKPOINT_BASE, "checkpoint-19830")


model = MBartForConditionalGeneration.from_pretrained(
    RESUME_CHECKPOINT,
    local_files_only=True    
)
model = model.to("cuda")

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8)
print("Data collator created")

loader = DataLoader(
    combined["train"].select(range(4)),
    batch_size=1, collate_fn=data_collator)
model.train()
batch = next(iter(loader))
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    out = model(**batch)
print(f"Sanity check loss: {out.loss.item():.4f}")
print(f"(Should be lower than 6.3 if checkpoint loaded correctly)")
assert not torch.isnan(out.loss), "NaN loss — stopping!"

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=combined["train"],
    eval_dataset=combined["valid"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

#clearing GPU
gc.collect()
torch.cuda.empty_cache()
used  = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {used:.1f} / {total:.1f} GB | Free: {total-used:.1f} GB")

trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")


In [ ]:
#Training Arguments
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from transformers import (Seq2SeqTrainingArguments,
                          Seq2SeqTrainer,
                          DataCollatorForSeq2Seq)

#OUTPUT_DIR = "/kaggle/working/cultural-recipes-mbart"  #note that this a different dir from mT5
OUTPUT_DIR = "/kaggle/working/cultural-recipes-mbart"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=3e-5,
    label_smoothing_factor=0.0,
    predict_with_generate=True,
    generation_max_length=128,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=2000,             # changed from 500 — saves less frequently bc memory
    load_best_model_at_end=False,
    fp16=False,
    logging_steps=50,
    save_total_limit=1,          # instead of 3 only keep 1 checkpoint
    report_to="none",
    gradient_checkpointing=True,
    optim="adafactor",
)

In [ ]:
#run if ever need to reload model

from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import torch

CHECKPOINT_PATH = "/kaggle/input/datasets/jlkeyx/cultural-recipes-checkpoint/cultural-recipes-mbart/checkpoint-19830"

model = MBartForConditionalGeneration.from_pretrained(
    CHECKPOINT_PATH,
    local_files_only=True
).to("cuda")

tokenizer = MBart50TokenizerFast.from_pretrained(
    CHECKPOINT_PATH,
    local_files_only=True
)

model.eval()

In [2]:
#model evaluation (after successful final training)

model.eval()

def adapt_recipe(recipe_text, max_length=256):
    inputs = tokenizer(
        recipe_text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=5,
        early_stopping=True,
        no_repeat_ngram_size=4
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

en_recipe = (
    "Title: Scrambled Eggs, "
    "Ingredients: 3 eggs, 2 tablespoons butter, salt, pepper, "
    "Steps: Crack eggs into bowl. Whisk well. "
    "Melt butter in pan over medium heat. "
    "Add eggs and stir gently until cooked."
)

cn_recipe = (
    "Title: 番茄炒蛋, "
    "Ingredients: 3个鸡蛋, 2个番茄, 盐, 糖, 食用油, "
    "Steps: 鸡蛋打散。番茄切块。热锅加油，炒鸡蛋至凝固，"
    "加入番茄翻炒，加盐和糖调味即可。"
)

print("=== TEST 1: English → Chinese ===")
print("INPUT: ", en_recipe[:80], "...")
print("OUTPUT:", adapt_recipe(en_recipe))

print("\n=== TEST 2: Chinese → English ===")
print("INPUT: ", cn_recipe[:80], "...")
print("OUTPUT:", adapt_recipe(cn_recipe))

=== TEST 1: English → Chinese ===
INPUT:  Title: Scrambled Eggs, Ingredients: 3 eggs, 2 tablespoons butter, salt, pepper,  ...
OUTPUT: Title: Scrambled Eggs, Ingredients: 4 eggs, 1 Tbsp. butter, salt and pepper to taste, Steps: Melt butter in skillet. Add eggs and scramble. Season with salt and pepper.

=== TEST 2: Chinese → English ===
INPUT:  Title: 番茄炒蛋, Ingredients: 3个鸡蛋, 2个番茄, 盐, 糖, 食用油, Steps: 鸡蛋打散。番茄切块。热锅加油，炒鸡蛋至凝固，加入 ...
OUTPUT: Title: Deviled Eggs With Tomatoes, Ingredients: 6 hard-cooked eggs, cut lengthwise in half, 1/4 cup KRAFT Real Mayo Mayonnaise, 2 Tbsp. CLAUSSEN Sweet Pickle Relish, 2 tsp. GREY POUPON Dijon Mustard, Steps: Mash egg yolks in medium bowl with fork. Add mayo, relish and mustard; mix well. Fill whites with yolk mixture.


In [ ]:
#saving checkpoint for next session and google colab import

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath) / 1024 / 1024
        if size > 0.1:  # only show files > 0.1MB
            print(f"  {fpath}  ({size:.1f} MB)")
